# Assignment 5: Reinforcement Learning

In this assignment we will apply Reinforcement Learning (RL) to some benchmark tasks. The assignment uses torch to implement neural networks that represent the Q-function and the *gymnasium* environment to model the benchmark tasks. *gymnasium* is a python package that allows to implement Markov Decision Processes (MDPs). Each MDP is implemented as an environment class that provides interfaces 
 * to reinitialize the environment randomly in an initialization area
 * to perform a transition from the present state applying a certain action to a successor state which also provides an immediate reward
 * to control whether a terminal state has been reached or a maximal number of transitions has been executed in the present episode
 * to visualize the environment graphically
 * to interact with the environment using the keyboard to control the system

We will work with gymnasium using the two benchmarks *Cart-Pole* and *Mountain-Car*.

First of all, we have to import the required packages and initialize them. If you did not yet have all necessary packages installed, use *pip install* to get them.

In [ ]:
%pip install gymnasium
%pip install pygame
%pip install PyQt5
%pip install matplotlib

import collections
import random
import torch
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import gymnasium as gym
import math
from gymnasium.utils import play

%gui qt5
#%matplotlib qt        # if you get an error message from the line above, you might try this instead. Unfortunatly, plotting 3d plots and interactive plots is nasty in jupyter notebooks
#%matplotlib inline

# if GPU is to be used
device  = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print (f'using device {device}')

## 1. Understanding the code of a generic RL-agent

First, we want to implement a generic RL-agent following the basic ideas of Deep-Q-networks (DQN). The implementation itself is provided by us while your task is to understand and use it. Hence, look at the following code snippets and answer the following questions:
 * what does a Transition model?
 * what does a replay buffer store?
 * what happens if we push additional data to a replay buffer when the replay buffer is already filled completely?

In [ ]:
# transition is a tuple (s, a, s', r)
Transition = collections.namedtuple('Transition', ('state', 'action', 'reward', 'next_state'))

# a simple replay buffer organized as a queue
class ReplayBuffer(object):

    # create a simple queue to store transitions
    def __init__(self, capacity):
        self.memory = collections.deque([], maxlen=capacity)

    # push new transitions into the queue. Remove old ones if capacity is too small
    def push(self, state, action, reward, next_state):
        """Save a transition"""
        self.memory.append(Transition(state, action, reward, next_state))

    # create a random sample without replacement from the replay memory
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    # povide size of buffer
    def __len__(self):
        return len(self.memory)

    # a pruning strategy to remove examples from the replay buffer.
    # The pruning strategy partitions the state-action space into cells
    # and samples examples considering how many examples are in a
    # cell. Hence, examples from areas with only few examples are
    # resampled more frequently than examples for areas with lots
    # of examples.
    #  'num_keep' is the number of samples to keep
    #  'minima' is an array that contains the minimal values for each state dimension
    #  'maxima' is an array that contains the maximal values for each state dimension
    def pruning (self, num_keep, minima, maxima, num_actions):
        if num_keep>len(self.memory):
            return
        bins = 10
        dims = len(minima)
        minima = np.array(minima, dtype=np.float64)
        maxima = np.array(maxima, dtype=np.float64)
        index = np.empty((dims+1,), dtype=np.int8)
        index[1:] = bins
        index[0] = num_actions
        accumulator = np.zeros(shape=index)
        diffs = (maxima-minima)/bins
        for transition in self.memory:
            index[1:] = np.minimum(np.maximum(np.floor((transition.state-minima)/diffs),0),bins-1).squeeze()
            index[0] = transition.action
            accumulator[tuple(index)] += 1
        nzf = 1.0/np.count_nonzero(accumulator)
        selection_probs = []
        for transition in self.memory:
            index[1:] = np.minimum(np.maximum(np.floor((transition.state-minima)/diffs),0),bins-1).squeeze()
            index[0] = transition.action
            selection_probs.append(nzf/accumulator[tuple(index)])
        selectors = np.r_[0:len(self.memory)]
        selected = np.array(np.random.choice (selectors, num_keep, replace=False, p=selection_probs), dtype=np.int32)
        new_memory = collections.deque([], maxlen=self.memory.maxlen)
        for index in selected:
            new_memory.append(self.memory[index])
        self.memory = new_memory

Consider also the implementation of the class DQN and answer the following questions:
 * which kind of neural network does a DQN represent?
 * how many input neurons and how many output neurons does it have?
 * Which activation function do the output neurons use, which activation function do the hidden neurons use?
 * which function do we want to represent with a DQN? How do we apply it?
 * which function do we have to call when we want to get the best action in a certain state of the MDP?
 * how do I have to call DQN if I want to build a neural network with 5 hidden layers, each composed out of 14, 20, 35, 25, and 12 neurons? The state vector should be composed of 3 state variables and the action set should contain 5 actions.

In [ ]:
# a deep Q network
class DQN(torch.nn.Module):

    # create a network with the given structure.
    # all hidden layers are fully connected and use ReLu activation
    # the output layer is fully connected and uses linear activation
    #  'n_observation' describes the number of inputs
    #  'n_action' describes the number of outputs (one for each action)
    #  'n_hidden' is a list. Each entry describes the number of neurons in a hidden layer
    def __init__(self, n_observations, n_actions, n_hidden):
        super(DQN, self).__init__()
        self.layers = torch.nn.Sequential().to(device)
        n_prev = n_observations
        for n in n_hidden:
            self.layers.append(torch.nn.Linear(n_prev, n))
            self.layers.append(torch.nn.ReLU())
            n_prev = n
        self.layers.append (torch.nn.Linear(n_prev, n_actions))
    
    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        return self.layers(x)

    # return the index of the most probable action in the given state
    def get_action(self, state):
        with torch.no_grad():
            return self.forward(state).argmax(1)

    # return the maximal Q-value in the given state, e.g. evaluate the value function
    def get_value(self, state):
        with torch.no_grad():
            return self.forward(state).max(1).values
            

The following two functions are heuristics to control the benchmark tasks. At the moment, you might ignore them. Later on, you may implement them in a reasonable manner to control the benchmark tasks (see section 3.2)

In [ ]:
# heuristics for cart-pole
def heuristics_cart_pole (state):
    action = torch.tensor([0], dtype=torch.int64, device=device)

    #
    # fill in your code here
    #
    
    return action    

# heuristics for mountain car
def heuristics_mountain_car (state):
    action = torch.tensor([0], dtype=torch.int64, device=device)

    #
    # fill in your code here
    #
    
    return action
    

The class RL_Training serves a kind of template that allows to implement different RL algorithms. It loads the benchmark environments, setsup some settings to operate them in an appropriate manner, implements a generic control loop and provides some auxiliary functions to visualize data. To implment a specific RL-algorithm you need to derive from RL_Training and reimplement the functions `setup_rlagent()`, `get_best_action()`, `get_value()`, and `optimize_model()`. An example of such a specific algorithm is found in class DQN_Training below.

First focus on the function `run_test_trials()` which implement the execution of a test trial with an already learnt policy. Look at the different steps and answer the following questions:
 * in which step do we evaluate the learnt policy?
 * in which step does the environment change its state?
 * when does the control loop terminate?

Now, focus on the function `run_training_trials()` which implements the execution of training trials and modifies the policy. Compare the two functions `run_test_trials()` and `run_training_trials()` and answer the following questions:
 * which parts are the same, which parts are differing between training and testing?
 * in which way does the action selection differ?
 * at which point will the policy be modified?

In [ ]:
# an abstract class to implement Reinforcement Learning.
# derive from it to implement a specific learning algorithm
# and overwrite the functions setup_rlagent(), get_best_action(),
# get_value(), and optimize_model().
#
# If you want to use this class
# 1. generate an instance of a subclass of it for the respective scenario
# 2. train the model calling run_training_trials()
# 3. if you want to add experience with manual interaction call run_interactive_trials()
# 4. if you want to add experience generated with a heursitics, implement your 
#    heuristics in heuristics_cart_pole() or heuristics_mountain_car() and run
#    run_heuristic_trials()
# 5. if you want to see the learnt policy in operation, execute run_test_trials()
# 6. if you want to visualize the policy, call plot_value_function() and plot_replay_buffer()
#    Both work best for 2d state spaces as in the Mountain Car scenario
class RL_Training:

    # initialization defines the scenario and some parameters
    #  'plant' decribes the scenario, either 'MountainCar' or 'CartPole'
    #  'replay_buffer_size' defines the maximal capacity of the replay buffer
    #  'n_hidden' defines the number of hidden layers and neurons per hidden layer
    #  'discount' defines the discount rate
    #  'maximal_epsilon' defines the exploration rate at the very beginning of the training experiments
    #  'minimal_epsilon' defines the lower bound for the exploration rate
    #  'epsilon_halflife' defines the time until which the exploration rate epsilon has reached the middle between maximal_epsilon and minimal_epsilon (expnential decrease)
    def __init__ (self, plant, replay_buffer_size, n_hidden, discount=0.99, maximal_epsilon=0.9, minimal_epsilon=0.1, epsilon_halflife=2000):
        if plant=="MountainCar":
            self.plantname = "MountainCar-v0"
            self.action_remapping = { 0:1, 1:0, 2:2 }
            self.max_episode_steps = 300
            self.action_names = { 0:'nop', 1:'left', 2:'right' }
            self.state_names = { 0:'position', 1:'velocity' }
            self.state_min = [ -1.2, -0.07 ]
            self.state_max = [  0.6,  0.07 ]
            self.key_to_action = { 'a':1, 's':0, 'd': 2 }
            self.heuristics = heuristics_mountain_car
        elif plant=="CartPole":
            self.plantname = "CartPole-v1"
            self.action_remapping = { 0:0, 1:1 }
            self.max_episode_steps = 500
            self.action_names = { 0:'left', 1:'right' }
            self.state_names = { 0:'cart-pos', 1:'cart-vel', 2:'pole-angle', 3:'pole-vel' }
            self.state_min = [ -4.8, -2.5, -0.418, -2.0 ]
            self.state_max = [  4.8,  2.5,  0.418,  2.0 ]
            self.key_to_action = { 'a':0, 's':1 }
            self.heuristics = heuristics_cart_pole
        else:
            print(f'unknown or unsupported plant {plant}')
        
        self.setup_environment (None)
        self.discount = discount
        self.replay_buffer = ReplayBuffer (replay_buffer_size)
        self.n_hidden = n_hidden
        self.n_actions = self.env.action_space.n
        self.n_state = len(self.env.observation_space.high)
        self.maximal_epsilon = maximal_epsilon
        self.minimal_epsilon = minimal_epsilon
        self.epsilon_decay = epsilon_halflife/math.log(2)
        self.setup_rlagent ()
        self.figure_value_function = None
        self.figure_replay_buffer = None
        self.figure_training_rewards = None
        self.axes_value_function = None
        self.axes_replay_buffer = None
        self.axes_training_rewards = None

    # function to initialize the environment
    # will be called from inside the class, you don't need to call it explicitely
    #  'render_mode': either None (no visualization), 'human' or 'rgb_array' (with visualization)
    def setup_environment (self, render_mode):
        self.env = gym.make(self.plantname, render_mode=render_mode, max_episode_steps=self.max_episode_steps)
        self.env = gym.wrappers.TransformAction(self.env, lambda a: self.action_remapping[a], self.env.action_space)

    # function to initialize the reinforcement learning method
    # will be called from inside the class, you don't need to call it explicitely
    # overwrite it for your own learning algorithm
    def setup_rlagent (self):
        self.episode_rewards = []
        self.steps_done = 0

    # function to optimize the model, i.e. to learn from experience
    # function is called from run_training_trials(), you don't need to call it explicitely from outside
    # overwrite this function for your own learning algorithm
    def optimize_model(self, done):
        pass

    # function that provides the best action according to the learned policy
    # overwrite this function for your own learning algorithm
    def get_best_action(self, state):
        return torch.tensor([self.env.action_space.sample()], device=device, dtype=torch.long)

    # function that evaluates the values function V(s) of the learnt policy
    # overwrite this function for your own learning algorithm
    def get_value(self, state):
        return 0.0
    
    # return the index of an action following an epsilon-greedy strategy
    def get_epsilon_greedy_action(self, epsilon):
        rv = random.random()
        if rv > epsilon:
            return self.get_best_action(self.state)
        else:
            return torch.tensor([self.env.action_space.sample()], device=device, dtype=torch.long)

    # a callback function to add transitions to the replay buffer.
    # You don't need to call it explicitely from outside,
    # it is called from run_interactive_trials()
    def memorize_transitions_callback (self, obs_t, obs_tp1, action, rew, terminated, truncated, info):
        try:
            state = torch.tensor(obs_t, dtype=torch.float32, device=device).unsqueeze(0)
            next_state = torch.tensor(obs_tp1, dtype=torch.float32, device=device).unsqueeze(0)
            action1 = torch.tensor(action, dtype=torch.int64, device=device).unsqueeze(0)
            reward = torch.tensor([rew], dtype=torch.float32, device=device)
            if terminated:
                next_state = None
            self.replay_buffer.push(state, action1, reward, next_state)
        except Exception as e: 
            pass

    # Execute interactive trials of the scenario. 
    # A visualization of the scenario pops up and you can control the
    # agent with keyboard commands. Try 'a', 's', 'd', etc.
    # Pressing 'ESC' quits the interactive session.
    # Transitions during interactive trials are recorded and added to the replay buffer.
    def run_interactive_trials(self):
        self.setup_environment("rgb_array")
        play.play(self.env, callback=self.memorize_transitions_callback, keys_to_action=self.key_to_action)

    # Execute a heuristic control strategy for a scenario. 
    # If visualization is set to "human" a visualization of the scenario pops up.
    # You can define the heuristics to use by modifying the functions 
    # 'heuristics_cart_pole()' and 'heuristics_mountain_car()' from above.
    # Transitions during heuristic trials are recorded and added to the replay buffer.
    def run_heuristic_trials(self, num_episodes=1, visualization=None):
        self.setup_environment (visualization)

        # for each episode, do
        for i_episode in range(num_episodes):
            state1, info = self.env.reset()
            self.state = torch.tensor(state1, dtype=torch.float32, device=device).unsqueeze(0)
            episode_over = False
            sum_of_rewards = 0.0
            steps = 0
            while not episode_over:
                action = self.heuristics(self.state)
                new_state, reward, terminated, truncated, _ = self.env.step(action.item())
                sum_of_rewards = self.discount*sum_of_rewards+reward
                steps += 1
                
                episode_over = terminated or truncated
                reward = torch.tensor([reward], device=device)

                if terminated:
                    next_state = None
                else:
                    next_state = torch.tensor(new_state, dtype=torch.float32, device=device).unsqueeze(0)
                
                # Store the transition in memory
                self.replay_buffer.push(self.state, action, reward, next_state)

                # Move to the next state
                self.state = next_state

            if truncated:
                print(f'episode terminated after reaching maximal number of {steps} steps with reward {sum_of_rewards}')
            else:
                print(f'episode terminated after {steps} steps with reward {sum_of_rewards}')
    
    # Execute a trial evaluation the learnt policy. 
    # If visualization is set to "human" a visualization of the scenario pops up.
    # Transitions during test trials are **not** added to the replay buffer.
    def run_test_trials(self, num_episodes=1, visualization=None):
        self.setup_environment (visualization)

        # for each episode, do
        for i_episode in range(num_episodes):
            state1, info = self.env.reset()  # initialize a new epsisode by generating a random initial state
            self.state = torch.tensor(state1, dtype=torch.float32, device=device).unsqueeze(0)
            episode_over = False
            sum_of_rewards = 0.0
            steps = 0

            # the main control loop
            while not episode_over:
                action = self.get_best_action(self.state)
                new_state, reward, terminated, truncated, _ = self.env.step(action.item())
                sum_of_rewards = self.discount*sum_of_rewards+reward
                steps += 1
        
                episode_over = terminated or truncated
                self.state = torch.tensor(new_state, dtype=torch.float32, device=device).unsqueeze(0)

            if truncated:
                print(f'episode terminated after reaching maximal number of {steps} steps with reward {sum_of_rewards}')
            else:
                print(f'episode terminated after {steps} steps with reward {sum_of_rewards}')
    
    # Execute training runs for a scenario without any visualization.
    # Actions are chosen according to an epsilon-greedy strategy.
    # After each transition optimize_model() is called. If an episode is over
    # optimize_model(true) is called, otherwise optimize_model(false).
    # Transitions during training trials are recorded and added to the replay buffer.
    def run_training_trials(self, num_episodes):
        self.setup_environment (None)
        for i_episode in range(num_episodes):
            state1, info = self.env.reset()
            self.state = torch.tensor(state1, dtype=torch.float32, device=device).unsqueeze(0)
            episode_over = False
            sum_of_rewards = 0.0

            # the main control loop
            while not episode_over:
                epsilon = self.minimal_epsilon + (self.maximal_epsilon - self.minimal_epsilon) * math.exp(-1. * self.steps_done / self.epsilon_decay)
                action = self.get_epsilon_greedy_action(epsilon)
                new_state, reward, terminated, truncated, _ = self.env.step(action.item())
                sum_of_rewards = self.discount*sum_of_rewards+reward
                self.steps_done += 1

                episode_over = terminated or truncated
                reward = torch.tensor([reward], device=device)

                if terminated:
                    next_state = None
                else:
                    next_state = torch.tensor(new_state, dtype=torch.float32, device=device).unsqueeze(0)

                # Store the transition in memory
                self.replay_buffer.push(self.state, action, reward, next_state)

                # Move to the next state
                self.state = next_state

                # Perform one step of the optimization
                self.optimize_model(episode_over)

            self.episode_rewards.append(sum_of_rewards)
            self.plot_rewards(self.episode_rewards)

    # a callback function for internal purposes. Don't call it from outside.
    def on_close_figure_training_rewards (self, event):
        self.figure_training_rewards = None

    # Generate a 2d plot of the reward obtained during the training runs so far
    def plot_rewards (self, rewards):
        if self.figure_training_rewards==None:
            self.figure_training_rewards = plt.figure()
            self.figure_training_rewards.canvas.mpl_connect('close_event', self.on_close_figure_training_rewards)
            self.axes_training_rewards = self.figure_training_rewards.add_subplot(111)
        self.axes_training_rewards.clear()
        self.axes_training_rewards.set_title('training rewards')
        self.axes_training_rewards.set_xlabel('episode')
        self.axes_training_rewards.set_ylabel('reward')
        self.axes_training_rewards.plot(np.array(rewards))
        self.figure_training_rewards.canvas.draw()
        self.figure_training_rewards.canvas.flush_events()
        plt.show(block=False)
        
    # a callback function for internal purposes. Don't call it from outside.
    def on_close_figure_value_function (self, event):
        self.figure_value_function = None
    
    # Generate a 3d plot of the learnt value function V(s).
    # Works best if the state space is 2-dimensional. Otherwise, select the
    # two state dimensions that you'd like to visualize and provide a 
    # prototype state vector with fixed values for all the other state dimensions
    def plot_value_function (self, prototype_state=None, dim1=0, dim2=1):
        if self.figure_value_function==None:
            self.figure_value_function = plt.figure()
            self.figure_value_function.canvas.mpl_connect('close_event', self.on_close_figure_value_function)
            self.axes_value_function = self.figure_value_function.add_subplot(111, projection='3d')
        x = np.arange(self.state_min[dim1], self.state_max[dim1], (self.state_max[dim1]-self.state_min[dim1])/20)
        y = np.arange(self.state_min[dim2], self.state_max[dim2], (self.state_max[dim2]-self.state_min[dim2])/20)
        X, Y = np.meshgrid(x, y)
        Z = np.zeros(shape=X.shape)
        if prototype_state==None:
            prototype_state = torch.zeros([1,2], dtype=torch.float32, device=device)
        for i in range(Z.shape[0]):
            for j in range(Z.shape[1]):
                prototype_state[0,dim1] = X[i,j]
                prototype_state[0,dim2] = Y[i,j]
                with torch.no_grad():
                    Z[i,j] = self.get_value(prototype_state)
        self.axes_value_function.clear()
        self.axes_value_function.set_title('value function')
        self.axes_value_function.set_xlabel(self.state_names[dim1])
        self.axes_value_function.set_ylabel(self.state_names[dim2])
        self.axes_value_function.set_zlabel('V(s)')
        self.axes_value_function.set_xlim(self.state_min[dim1], self.state_max[dim1])
        self.axes_value_function.set_ylim(self.state_min[dim2], self.state_max[dim2])
        self.axes_value_function.plot_surface(X, Y, Z, cmap=matplotlib.cm.Wistia, linewidth=0, antialiased=False)
        self.figure_value_function.canvas.draw()
        self.figure_value_function.canvas.flush_events()
        plt.show(block=False)

    # a callback function for internal purposes. Don't call it from outside.
    def on_close_figure_replay_buffer (self, event):
        self.figure_replay_buffer = None
    
    # Generate a 3d scatter plot of the transitions in the replay buffer.
    # Works best if the state space is 2-dimensional. Otherwise, select the
    # two state dimensions that you'd like to visualize and provide a 
    # prototype state vector with fixed values for all the other state dimensions.
    # Each transition is placed at the respective point in state space. The
    # height of each point refers to the estimated value V(s). The color of the
    # point refers to arg max_a Q(s,a), i.e. it does not encode the action of 
    # the recorded transition but the action that would be taken by the current
    # policy.
    def plot_replay_buffer (self, prototype_state=None, dim1=0, dim2=1):
        if self.figure_replay_buffer==None:
            self.figure_replay_buffer = plt.figure()
            self.figure_replay_buffer.canvas.mpl_connect('close_event', self.on_close_figure_replay_buffer)
            self.axes_replay_buffer = self.figure_replay_buffer.add_subplot(111, projection='3d')
        if prototype_state==None:
            prototype_state = torch.zeros([1,2], dtype=torch.float32, device=device)
        xs = np.zeros(shape=[len(self.replay_buffer)])
        ys = np.zeros(shape=[len(self.replay_buffer)])
        zs = np.zeros(shape=[len(self.replay_buffer)])
        acts = [0] * len(self.replay_buffer)
        for i in range(len(self.replay_buffer)):
            xs[i] = self.replay_buffer.memory[i].state[0,dim1].item()
            ys[i] = self.replay_buffer.memory[i].state[0,dim2].item()
            prototype_state[0,dim1] = xs[i]
            prototype_state[0,dim2] = ys[i]
            with torch.no_grad():
                zs[i] = self.get_value(prototype_state)
                acts[i] = self.get_best_action(prototype_state)
                
        self.axes_replay_buffer.clear()
        self.axes_replay_buffer.set_title('trajectories in state space')
        self.axes_replay_buffer.set_xlabel(self.state_names[dim1])
        self.axes_replay_buffer.set_ylabel(self.state_names[dim2])
        self.axes_replay_buffer.set_zlabel('V(s)')
        self.axes_replay_buffer.set_xlim(self.state_min[dim1], self.state_max[dim1])
        self.axes_replay_buffer.set_ylim(self.state_min[dim2], self.state_max[dim2])
        
        cm = matplotlib.cm.hsv
        for a in range(self.env.action_space.start, self.env.action_space.start+self.env.action_space.n):
            mask = torch.tensor(tuple(map(lambda s: s==a, acts)), device=device, dtype=torch.bool)
            color = (1.0*(a-self.env.action_space.start))/self.env.action_space.n
            self.axes_replay_buffer.scatter(xs[mask], ys[mask], zs[mask], s=1, color=cm(color), label=self.action_names[a])
        self.axes_replay_buffer.legend()
        self.figure_replay_buffer.canvas.draw()
        self.figure_replay_buffer.canvas.flush_events()
        plt.show(block=False)
        

The class *DQN_Training* implements a simple version of the Deep-Q-ntworks approach. Go through the code and answer the following questions
 * in which way does it determine the best action for a state?
 * what is the role of policy_net and target_net in the implementation?
 * why is it necessary to distinguish between terminal states (states for which next_state=None) and non-terminal states during training? How are the treated differently?

In [ ]:
# A specialization of RL_Training that implements one simple form of (Deep) Q-Networks
# The method maintains two networks with identical structure, policy_net and
# target_net. policy_net is used to determine the best action and it is incrementally
# modified after each interaction with the environment. traget_net is used to calculate
# the target values for training. It is updated from policy_net by calculating 
# exponential moving averages over the network parameters. This means, target_net 
# changes much slower than policy_net. This helps to obtain stable learning.
class DQN_Training(RL_Training):
    # initialize DQN training
    #  'plant' decribes the scenario, either 'MountainCar' or 'CartPole'
    #  'replay_buffer_size' defines the maximal capacity of the replay buffer
    #  'n_hidden' defines the number of hidden layers and neurons per hidden layer
    #  'discount' defines the discount rate
    #  'batch_size' defines the batch size of the random sample for incremental learning steps
    #  'adam_learning_rate' defines the learning rate for Adam
    #  'rl_learning_rate' defines the learning rate for updating the target_net
    def __init__ (self, plant, replay_buffer_size, n_hidden, discount=0.99, maximal_epsilon=0.9, minimal_epsilon=0.1, epsilon_halflife=2000, batch_size=256, adam_learning_rate=1e-4, rl_learning_rate=0.01):
        self.batch_size = batch_size
        self.adam_learning_rate = adam_learning_rate
        self.rl_learning_rate = 0.01
        super().__init__(plant, replay_buffer_size, n_hidden, discount, maximal_epsilon, minimal_epsilon, epsilon_halflife)

    def setup_rlagent (self):
        super().setup_rlagent ()
        self.policy_net = DQN(self.n_state, self.n_actions, self.n_hidden).to(device)
        self.target_net = DQN(self.n_state, self.n_actions, self.n_hidden).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.policy_net.eval()
        self.optimizer = torch.optim.AdamW(self.policy_net.parameters(), lr=self.adam_learning_rate, amsgrad=True)

    def get_best_action(self, state):
        return self.policy_net.get_action(state)
        
    def get_value(self, state):
        return self.policy_net.get_value(state)

    def optimize_model(self, done):
        if len(self.replay_buffer) < self.batch_size:
            return
            
        self.policy_net.train()

        # get a random batch of transitions
        transitions = self.replay_buffer.sample(self.batch_size)
        batch = Transition(*zip(*transitions))

        # compute a mask of non-final states and concatenate the batch elements
        non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
        non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
        state_batch = torch.cat(batch.state)
        action_batch = torch.cat(batch.action).unsqueeze(1)
        reward_batch = torch.cat(batch.reward)

        # compute Q(s_t, a) from policy_net
        state_action_values = self.policy_net(state_batch).gather(1, action_batch)

        # compute V(s_{t+1}) for all next states from target_net
        # for final states set V(s_{t+1})=0, i.e. we treat the immediate reward as termination reward
        next_state_values = torch.zeros(self.batch_size, device=device)
        next_state_values[non_final_mask] = self.target_net.get_value(non_final_next_states)
        # compute the expected Q values
        expected_state_action_values = (next_state_values * self.discount) + reward_batch

        # compute loss
        criterion = torch.nn.SmoothL1Loss()
        loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

        # optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        # in-place gradient clipping
        torch.nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
        self.optimizer.step()
        
        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = self.target_net.state_dict()
        policy_net_state_dict = self.policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*self.rl_learning_rate + target_net_state_dict[key]*(1-self.rl_learning_rate)
        self.target_net.load_state_dict(target_net_state_dict)
        self.policy_net.eval()


## 2. Making our first steps in RL

With the help of DQN we can train our first controller with RL. For this we are using the cart-pole environment. Imagine a cart-pole system as a cart that is moving on a straight track. We can accelerate the cart to the left or to the right by appliing one the two available actions 0 (left) and 1 (right). A pole is mounted on the cart. The pole can rotate around a horizontal axis. We don't have direct control to the pole but we can only influence the pose of the pole by accelerating the cart left or right. Our goal is to balance the pole in upright position. We will obtain a reward of +1 if the pole is in upright position and a reward of 0 otherwise.

In the given implementation of the cart-pole system only zenith angles close to 0 (=upright) are considered. Whenever the zenith angle is outside of the interval from -24 to +24 degree the episode ends. As well, the episode ends when the cart leaves the track on the left or on the right or if the episode takes longer than 500 iterations.

The state space of the cart-pole system is four-dimensional. The state variables are
 * the position of the cart along the track
 * the velocity of the cart
 * the zenith angle of the pole
 * the angular velocity of the pole
The environment allows you access to all four variables which are arranges in the state vectors.

The following code creates a new instance of DQN_Training for the cart-pole-system.

In [ ]:
rl = DQN_Training (plant="CartPole", replay_buffer_size=12500, n_hidden=[ 128, 128, 128, 128 ], batch_size=256)

Train the DQN running run_training_trials() for multiple episodes. Check the performance of the trained network with run_test_trials().
 * Which learning behavior do you observe? Does the system get better over time or worse? What happens when you are training several hundreds of episodes?
 * What happens when you are changing the network size? How many hidden layers and how many neurons per layer provide reasonable learning behavior?
 * Whenever the learning behavior does not evolve as expected, do you find an explanation for it?
 * Hint: in RL applications overfitting is usually not a critical topic. So you don't need to spend time on fine tuning regularization techniques.

## 3. Understanding and improving its performance

### 3.1 Visualizing the state space

After having made our first experiences with RL we want to understand it closer by analyzing the state space. Since the four-dimensional state space of the cart-pole system does not allow a nice visualization we look at the Mountain-car scenario.

Mountain-car is a scenario which models an underactuated car that should drive upon a hill. It starts in a valley and can accelerate forwards (to the right, action=2) or backwards (to the left, action=1) or do not accelerate at all (action=0). The motor is not strong enough to drive straight on so that the car has to swing forth and back to be able to reach its goal on the top of the mountain. The car achieves a reward of -1 for each step until it reaches the goal position at which the episode terminated. Hence, the challenge is to reach the goal position as quickly as possible. The state space is two-dimensional and consists of the position of the car and its velocity.

In the simulation environment the car cannot exceed the boundaries of the track, i.e. whenever it touches the boundary it will just be blocked in moving further but the episode is neither truncated nor does the car get negative reward for touching the boundary. Each episode is truncated after at most 300 transitions.

Try to train an RL-controller for mountain car environment (plant="MountainCar") and execute test runs to see the peformance of the trained policy.
 * How well can RL train the task?
 * If RL cannot train a reasonable policy, why?
 * Visualize the learnt value function $V(s)$ calling the function `rl.plot_value_function()`
 * Visualize the content of the replay buffer calling the function `rl.plot_replay_buffer()`. The function will visualize all collected transitions in the state space. The height of the points refers to the learnt value function $V(s)$, the color indicates which action would be chosen in this state following the learnt policy.
 * Check whether the learnt value function is reasonable.
 * Check which part of the state space is filled with collected transitions.

### 3.2 Getting better trajectories during training

One issue in the mountain car example is that the state space can hardly be sampled completely as long as we are always starting in the valley and we didn't yet learn how to swing up. Since we always obtain a reward of -1 and the episodes always last 300 steps as long as we are unable to reach the goal, reinforcement learning will never be able to learn a reasonable strategy.

One way to overcome this issue would be to sample experience from the whole state space, i.e. to set the initial position of the car to other places as the bottom of the valley. However, the simulation environment does not allow us to do so and, maybe, it would also not be a realistic scenario. Another way to overcome the problem is to generate successful trajectories that reach the goal state and so to demonstrate to the RL algorithm how the problem could be solved. The demonstration strategy does not necessarily be optimal, it just needs to reach the goal.

Generate demonstration trajectories in one of two ways
 * run the environment in manual mode and control the car yourself. You can start interactive trials calling the function `run_interactive_trials()`. You can control the car pressing the keys 'a' (accelerate left) and 'd' (accelerate right). If you don't press a key the motor is not accelerating the vehicle at all. You can quit the interactive mode pressing the Escape key. During the interactive trials all transitions are recorded and pushed into the replay buffer.
 * implement a heuristic controller for the mountain car scenario by implementing the function `heuristics_mountain_car()`. The function takes the two-dimensional state vector and return the action. The first entry in the state vector encodes the position, the second entry the velocity. The possible actions are 0 (no acceleration), 1 (accelerate left), and 2 (accelerate right). After having implemented the heuristic controller you can run it calling the function `run_heuristic_trials()`. During execution of the heursitics the transitions will be recorded and pushed into the replay buffer. If you prefer to observe how the heuristics interacts with the invironment you can visualize the environment during heuristic trials setting the visualization argument to "human".

Retrain the RL policy and observe whether the task can be learnt. Compare the performance of the heuristics trials and the test trials with the learnt policy. Visualize the value function and the content of the replay buffer.

### 3.3 Experience management

When you train the mountain car or the cart-pole system you might realize that RL tends to forget an already learnt policy. Try to find explanations why.

A possible way to overcome the problem is to get a better management of the gathered experience in the replay buffer. In its standard settings the replay buffer has a maximal capacity. As soon as you push more data into the replay buffer it automatically removes the oldest transitions. Hence, the replay buffer only contains data from the most recent episodes which might not be sufficient to describe the shape of the Q-function in all areas of the joint state-action space. Some areas might be unpopulated or contain only a few data points. In subsequent optimization steps these unpopulated areas are ignored and so it is not guaranteed that the learnt Q-function is correct in these areas.

A possibile solution to overcome or at least reduce this problem is to improve the management of the replay buffer. Rather than deleting the oldest transitions we would like to remove transitions in the highly populated areas of the joint state-action space while preserving transitions in the less populated areas. An appropriate method is implemented in the function `pruning()`of the ReplayBuffer class. Analyze the code of this function and describe how the removal of examples in the replay buffer is organized.

Extend your RL approach by pruning the replay buffer from time to time to allow new transitions to be added without removing the most valueable previous experience. Retrain the RL policy and observe the performance of the RL algorithm over time. After having learnt a policy for the mountain car scenario go back to the cart-pole environment and train an RL policy

### 3.4 Improving the learning algorithm

Besides getting better training trajectories and optimizing the management of past experience another possibility to improve RL is to improve the learning steps themselves. Instead of executing tiny little modifications of the Q-network we aim to have larger updates from time to time. Hence, we are collecting data for several episodes without modifying the Q-network and then we perform several Q-iterations based on the collected experience. In each Q-iteration we are calculating the tuples of state, action, and expected future reward based on the present Q-network. Then, we retrain the Q-network. We can execute several of these Q-iterations in a row to speed up learning.

The class NFQ_Training implements this kind of reinforcement learning. Analyze the code and answer the following questions
 * what do the arguments *retraining_intervals* and *q_iterations* describe?
 * why don't we need a target_net as in DQN_Training?

Apply the NFQ_Training to the two benchmark scenarios cart-pole and mountain car and observe its behavior.

In [ ]:
# Another specialization of RL_Training that implements the (neural) fitted-Q-approach.
# The method maintains only one network policy_net that takes over the roles of policy_net
# and target_net in DQN_Training. In contrast to DQN_Training policy_net is used to
# to execute multiple episodes before it is modified. The modification is done executing
# several Q-iterations. In each Q-iteration target values for training are generated by
# evaluating policy_net for all collected transition samples and afterwards policy_net 
# is retrained in a supervised training manner for several iterations of Adam.
class NFQ_Training(RL_Training):
    # initialize NFQ training
    #  'plant' decribes the scenario, either 'MountainCar' or 'CartPole'
    #  'replay_buffer_size' defines the maximal capacity of the replay buffer
    #  'n_hidden' defines the number of hidden layers and neurons per hidden layer
    #  'discount' defines the discount rate
    #  'retraining_intervals' specifies how many episodes should be executed before policy_net is retrained
    #  'q_iterations' defines how many Q-iterations should be executed in a row
    #  'adam_learning_rate' defines the learning rate for Adam
    def __init__ (self, plant, replay_buffer_size, n_hidden, discount=0.99, maximal_epsilon=0.9, minimal_epsilon=0.1, epsilon_halflife=2000, retraining_interval=5, q_iterations=5, adam_learning_rate=1e-4):
        self.retraining_interval = retraining_interval
        self.adam_learning_rate = adam_learning_rate
        self.q_iterations = q_iterations
        super().__init__(plant, replay_buffer_size, n_hidden, discount, maximal_epsilon, minimal_epsilon, epsilon_halflife)

    def setup_rlagent (self):
        super().setup_rlagent ()
        self.policy_net = DQN(self.n_state, self.n_actions, self.n_hidden).to(device)
        self.episode_modulo_counter = 0
        self.optimizer = torch.optim.AdamW(self.policy_net.parameters(), lr=self.adam_learning_rate, amsgrad=True)

    def get_best_action(self, state):
        return self.policy_net.get_action(state)
        
    def get_value(self, state):
        return self.policy_net.get_value(state)

    def optimize_model(self, done):
        if not done:
            return  # train only at the end of episodes
        self.episode_modulo_counter += 1
        if (self.episode_modulo_counter<self.retraining_interval):
            return  # train only after self.retraining_interval episodes
        self.episode_modulo_counter = 0

        for i in range(self.q_iterations):
            # perform a certain number of Q-iterations
            
            # get a full batch of transitions
            batch = Transition(*zip(*self.replay_buffer.memory))
            batch_size = len(self.replay_buffer)

            # compute a mask of non-final states and concatenate the batch elements
            non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
            non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
            state_batch = torch.cat(batch.state)
            action_batch = torch.cat(batch.action).unsqueeze(1)
            reward_batch = torch.cat(batch.reward)

            # compute V(s_{t+1}) for all next states from target_net
            # for final states set V(s_{t+1})=0, i.e. we treat the immediate reward as termination reward
            next_state_values = torch.zeros(batch_size, device=device)
            next_state_values[non_final_mask] = self.policy_net.get_value(non_final_next_states)
            # compute the expected Q values
            expected_state_action_values = (next_state_values * self.discount) + reward_batch

            # compute loss
            criterion = torch.nn.MSELoss()

            # optimize the model
            self.policy_net.train()
            for iter_train in range(100):
                # perform a certain number of gradient descent steps
                state_action_values = self.policy_net(state_batch).gather(1, action_batch)
                loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
            self.policy_net.eval()
            